# Imports

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedShuffleSplit
from transformers import ASTFeatureExtractor, ASTForAudioClassification, Wav2Vec2FeatureExtractor, AutoModel, get_cosine_schedule_with_warmup
import torchaudio.transforms as T
from collections import Counter
import matplotlib.pyplot as plt
import timm
import wandb

warnings.filterwarnings('ignore')

# Config

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
STEMS_DIR = Path('/kaggle/input/datasets/gulshanbhalla/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems')
MASHUP_DIR = Path('/kaggle/input/datasets/gulshanbhalla/jan-2026-dl-gen-ai-project/messy_mashup/mashups')
ESC50_DIR = Path('/kaggle/input/datasets/gulshanbhalla/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio')
TEST_CSV = Path('/kaggle/input/datasets/gulshanbhalla/jan-2026-dl-gen-ai-project/messy_mashup/test.csv')
OUT_DIR = Path('/kaggle/working/')

GENRES = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
GENRE2ID = {g: i for i, g in enumerate(GENRES)}
ID2GENRE = {i: g for g, i in GENRE2ID.items()}

AST_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'
MERT_NAME = 'm-a-p/MERT-v1-95M'
SR = 24000
CLIP_SEC = 10.0
CLIP_SAMPLES = int(SR * CLIP_SEC)
BATCH_SIZE = 8
AST_EPOCHS = 20
MERT_EPOCHS = 12
CNN_EPOCHS = 12
LR_AST = 8e-5
LR_MERT = 1e-4
CNN_LR = 3e-4
AST_UNFREEZE = 3
MERT_UNFREEZE = 2
VAL_SPLIT = 0.1

SNR_MIN, SNR_MAX = 0, 15
NOISE_PROB = 0.6

OFFSETS = [0.0, 10.0, 20.0]
TTA_OFFSETS = [0.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 15.0, 18.0, 21.0, 25.0]

AST_CHECKPOINT = str(OUT_DIR / 'ast_best.pt')
MERT_CHECKPOINT = str(OUT_DIR / 'mert_best.pt')

# Weights and Biases

In [ ]:
wandb.login(key='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')
run = wandb.init(
    project='23f3002949-t12026',
    entity='ehsaasbhalla007-iit-madras',
    name='ast + mert',
    settings=wandb.Settings(start_method='thread')
)

# Data Preparation

In [ ]:
def blend_stems(track_dir, sr=SR, duration=CLIP_SEC, offset=0.0):
    stems = list(Path(track_dir).glob('*.wav'))
    mixed = None
    start_sample = int(offset * sr)
    end_sample = start_sample + int(duration * sr)
    for sp in stems:
        try:
            key = str(sp)
            y, _ = librosa.load(key, sr=sr, mono=True, duration=duration, offset=offset)
            w = random.uniform(0.3, 1.5) if mixed is not None else 1.0
            mixed = y if mixed is None else mixed[:len(y)] + w * y[:len(mixed)]
        except:
            pass
    if mixed is not None and np.max(np.abs(mixed)) > 0:
        mixed /= np.max(np.abs(mixed))
    return mixed

def overlay_noise(signal, noise, snr_db):
    n = np.tile(noise, int(np.ceil(len(signal)/len(noise))))[:len(signal)]
    ps = np.mean(signal**2) + 1e-9
    pn = np.mean(n**2) + 1e-9
    scale = np.sqrt(ps / pn / (10**(snr_db/10)))
    out = signal + scale * n
    if np.max(np.abs(out)) > 0:
        out /= np.max(np.abs(out))
    return out.astype(np.float32)

In [ ]:
ast_fe = ASTFeatureExtractor.from_pretrained(AST_NAME)
mert_fe = Wav2Vec2FeatureExtractor.from_pretrained(MERT_NAME)
spectrogram_transform = nn.Sequential(
    T.MelSpectrogram(sample_rate=SR, n_mels=128, n_fft=400, hop_length=240),
    T.AmplitudeToDB()
)
resampler_16k = T.Resample(orig_freq=24000, new_freq=16000)

In [ ]:
noise_clips = []
if ESC50_DIR and ESC50_DIR.exists():
    esc50_files = list(ESC50_DIR.rglob('*.wav'))
    for fp in esc50_files:
        try:
            y, _ = librosa.load(str(fp), sr=SR, mono=True)
            if len(y) > 1000: noise_clips.append(y)
        except: pass

In [ ]:
all_entries = []
genre_tracks = {i: [] for i in range(len(GENRES))}
for genre in GENRES:
    gdir = STEMS_DIR / genre
    if not gdir.exists():
        continue
    for track_dir in sorted(gdir.iterdir()):
        genre_tracks[GENRE2ID[genre]].append(str(track_dir))
        for off in OFFSETS:
            all_entries.append((str(track_dir), off, GENRE2ID[genre]))

random.shuffle(all_entries)
labels_all = [s[2] for s in all_entries]
sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SPLIT, random_state=SEED)
tr_idx, val_idx = next(sss.split(all_entries, labels_all))
train_split, val_split = [all_entries[i] for i in tr_idx], [all_entries[i] for i in val_idx]

# Dataset Architecture

In [ ]:
def mask_spectrogram(spec, freq_mask_param=24, time_mask_param=80, n_masks=2):
    spec = spec.clone()
    T, F = spec.shape
    for _ in range(n_masks):
        f = random.randint(1, freq_mask_param)
        f0 = random.randint(0, max(F - f, 0))
        spec[:, f0:f0 + f] = 0.0
    for _ in range(n_masks):
        t = random.randint(1, time_mask_param)
        t0 = random.randint(0, max(T - t, 0))
        spec[t0:t0 + t, :] = 0.0
    return spec

class DualStemDataset(Dataset):
    def __init__(self, samples, noise_clips=None, augment=True, genre_tracks=None):
        self.samples = samples
        self.noise_clips = noise_clips or []
        self.augment = augment
        self.genre_tracks = genre_tracks or {}

    def __len__(self): return len(self.samples)

    def _cross_stem_blend(self, label, offset):
        tracks = self.genre_tracks.get(label, [])
        if len(tracks) < 2: return None
        stem_names = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
        mixed = None
        for sn in stem_names:
            td = random.choice(tracks)
            try:
                s, _ = librosa.load(str(Path(td) / sn), sr=SR, mono=True, duration=CLIP_SEC, offset=offset)
                w = random.uniform(0.3, 1.5) if mixed is not None else 1.0
                mixed = s if mixed is None else mixed[:len(s)] + w * s[:len(mixed)]
            except: pass
        if mixed is not None and np.max(np.abs(mixed)) > 0:
            mixed /= np.max(np.abs(mixed))
        return mixed

    def __getitem__(self, idx):
        track_dir, offset, label = self.samples[idx]
        y = self._cross_stem_blend(label, offset) if (self.augment and random.random() < 0.5) else None
        if y is None: y = blend_stems(track_dir, offset=offset)
        if y is None: y = np.zeros(CLIP_SAMPLES, dtype=np.float32)

        if len(y) < CLIP_SAMPLES: y = np.pad(y, (0, CLIP_SAMPLES - len(y)), mode='wrap')
        else: y = y[:CLIP_SAMPLES]

        if self.augment and random.random() < 0.3:
            y = librosa.effects.time_stretch(y, rate=random.uniform(0.88, 1.12))
            y = y[:CLIP_SAMPLES] if len(y) > CLIP_SAMPLES else np.pad(y, (0, CLIP_SAMPLES - len(y)), mode='wrap')

        if self.augment and self.noise_clips and random.random() < NOISE_PROB:
            y = overlay_noise(y, random.choice(self.noise_clips), random.uniform(SNR_MIN, SNR_MAX))

        if self.augment and random.random() > 0.5:
            y = np.roll(y, random.randint(0, CLIP_SAMPLES // 4))

        mert_audio = mert_fe(y.astype(np.float32), sampling_rate=24000, return_tensors='pt')['input_values'].squeeze(0)
        
        y_16k = resampler_16k(torch.tensor(y.astype(np.float32))).numpy()
        ast_audio = ast_fe(y_16k, sampling_rate=16000, return_tensors='pt')['input_values'].squeeze(0)
        
        cnn_spec = spectrogram_transform(torch.tensor(y.astype(np.float32))).transpose(0, 1)
        if self.augment: cnn_spec = mask_spectrogram(cnn_spec)

        return {'mert_audio': mert_audio, 'ast_audio': ast_audio, 'cnn_spec': cnn_spec, 'labels': torch.tensor(label, dtype=torch.long)}

In [ ]:
train_set = DualStemDataset(train_split, noise_clips, augment=True, genre_tracks=genre_tracks)
val_set = DualStemDataset(val_split, noise_clips, augment=False)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

v1_f1 = [0.9890, 0.9890, 0.9556, 0.9773, 0.9091, 0.9888, 0.9149, 0.9362, 0.9655, 0.8636]
raw_w = [1.0 / f for f in v1_f1]
mean_w = sum(raw_w) / len(raw_w)
class_weights = torch.tensor([w / mean_w for w in raw_w], dtype=torch.float32).to(DEVICE)

In [ ]:
def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    index = torch.randperm(x.size()[0]).to(DEVICE)
    return (lam * x + (1 - lam) * x[index, :]), y, y[index], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# Model 1: AST Foundation

In [ ]:
model_ast = ASTForAudioClassification.from_pretrained(AST_NAME, num_labels=10, ignore_mismatched_sizes=True, label2id=GENRE2ID, id2label=ID2GENRE).to(DEVICE)
def set_ast_trainable(trainable):
    for name, param in model_ast.named_parameters():
        if 'classifier' not in name: param.requires_grad = trainable

set_ast_trainable(False)
opt_ast = torch.optim.AdamW(filter(lambda p: p.requires_grad, model_ast.parameters()), lr=LR_AST, weight_decay=0.01)
sched_ast = get_cosine_schedule_with_warmup(opt_ast, int(0.1 * AST_EPOCHS * len(train_loader)), AST_EPOCHS * len(train_loader))
loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

ast_f1 = 0.0
for epoch in range(1, AST_EPOCHS + 1):
    if epoch == AST_UNFREEZE:
        set_ast_trainable(True)
        opt_ast = torch.optim.AdamW([{'params': model_ast.audio_spectrogram_transformer.parameters(), 'lr': LR_AST * 0.1}, {'params': model_ast.classifier.parameters(), 'lr': LR_AST}], weight_decay=0.01)
        sched_ast = get_cosine_schedule_with_warmup(opt_ast, 0, (AST_EPOCHS - epoch + 1) * len(train_loader))

    model_ast.train()
    for batch in tqdm(train_loader, desc=f'AST Ep {epoch}', leave=False):
        iv, labels = batch['ast_audio'].to(DEVICE), batch['labels'].to(DEVICE)
        iv, y_a, y_b, lam = mixup_data(iv, labels)
        opt_ast.zero_grad()
        loss = mixup_criterion(loss_fn, model_ast(input_values=iv).logits, y_a, y_b, lam)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ast.parameters(), 1.0)
        opt_ast.step(); sched_ast.step()

    model_ast.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            iv, labels = batch['ast_audio'].to(DEVICE), batch['labels'].to(DEVICE)
            val_preds += model_ast(input_values=iv).logits.argmax(-1).cpu().tolist()
            val_labels += labels.cpu().tolist()
    ep_f1 = f1_score(val_labels, val_preds, average='macro')
    print(f'[AST] Epoch {epoch:02d} Val F1: {ep_f1:.4f}')
    wandb.log({'ast_epoch': epoch, 'ast_val_f1': ep_f1})
    if ep_f1 > ast_f1:
        ast_f1 = ep_f1
        torch.save(model_ast.state_dict(), AST_CHECKPOINT)

print(f'AST Training Complete | Best F1: {ast_f1:.4f}')
model_ast.cpu(); torch.cuda.empty_cache()

# Model 2: MERT Foundation

In [ ]:
class MERTClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.mert = AutoModel.from_pretrained(MERT_NAME, trust_remote_code=True)
        self.mert.gradient_checkpointing_enable()
        self.classifier = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_classes))
    def forward(self, raw_audio):
        return self.classifier(self.mert(raw_audio).last_hidden_state.mean(dim=1))

model_mert = MERTClassifier().to(DEVICE)
def set_mert_trainable(trainable):
    for name, param in model_mert.named_parameters():
        if 'classifier' not in name: param.requires_grad = trainable

set_mert_trainable(False)
opt_mert = torch.optim.AdamW(filter(lambda p: p.requires_grad, model_mert.parameters()), lr=LR_MERT, weight_decay=0.01)
sched_mert = get_cosine_schedule_with_warmup(opt_mert, int(0.1 * MERT_EPOCHS * len(train_loader)), MERT_EPOCHS * len(train_loader))

mert_f1 = 0.0
for epoch in range(1, MERT_EPOCHS + 1):
    if epoch == MERT_UNFREEZE:
        set_mert_trainable(True)
        opt_mert = torch.optim.AdamW([{'params': model_mert.mert.parameters(), 'lr': LR_MERT * 0.1}, {'params': model_mert.classifier.parameters(), 'lr': LR_MERT}], weight_decay=0.01)
        sched_mert = get_cosine_schedule_with_warmup(opt_mert, 0, (MERT_EPOCHS - epoch + 1) * len(train_loader))

    model_mert.train()
    for batch in tqdm(train_loader, desc=f'MERT Ep {epoch}', leave=False):
        iv, labels = batch['mert_audio'].to(DEVICE), batch['labels'].to(DEVICE)
        iv, y_a, y_b, lam = mixup_data(iv, labels)
        opt_mert.zero_grad()
        loss = mixup_criterion(loss_fn, model_mert(iv), y_a, y_b, lam)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_mert.parameters(), 1.0)
        opt_mert.step(); sched_mert.step()

    model_mert.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            iv, labels = batch['mert_audio'].to(DEVICE), batch['labels'].to(DEVICE)
            val_preds += model_mert(iv).argmax(-1).cpu().tolist()
            val_labels += labels.cpu().tolist()
    ep_f1 = f1_score(val_labels, val_preds, average='macro')
    print(f'[MERT] Epoch {epoch:02d} Val F1: {ep_f1:.4f}')
    wandb.log({'mert_epoch': epoch, 'mert_val_f1': ep_f1})
    if ep_f1 > mert_f1:
        mert_f1 = ep_f1
        torch.save(model_mert.state_dict(), MERT_CHECKPOINT)

print(f'MERT Training Complete | Best F1: {mert_f1:.4f}')
model_mert.cpu(); torch.cuda.empty_cache()

# Model 3: EfficientNet-B2

In [ ]:
class SpectrogramClassifierB2(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b2', pretrained=True, num_classes=num_classes, in_chans=1)
    def forward(self, x): return self.backbone(x)

cnn_model = SpectrogramClassifierB2().to(DEVICE)
cnn_optimizer = torch.optim.AdamW(cnn_model.parameters(), lr=CNN_LR, weight_decay=0.01)
cnn_scheduler = get_cosine_schedule_with_warmup(cnn_optimizer, int(0.1 * CNN_EPOCHS * len(train_loader)), CNN_EPOCHS * len(train_loader))

effnet_f1 = 0.0
for epoch in range(1, CNN_EPOCHS + 1):
    cnn_model.train()
    for batch in tqdm(train_loader, desc=f'CNN Ep {epoch}', leave=False):
        x, labels = batch['cnn_spec'].unsqueeze(1).to(DEVICE), batch['labels'].to(DEVICE)
        if x.size(2) > 256: x = x[:, :, random.randint(0, x.size(2) - 256):][: ,:, :256, :]
        x, y_a, y_b, lam = mixup_data(x, labels)
        cnn_optimizer.zero_grad()
        loss = mixup_criterion(loss_fn, cnn_model(x), y_a, y_b, lam)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(cnn_model.parameters(), 1.0)
        cnn_optimizer.step(); cnn_scheduler.step()
    
    cnn_model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            x, labels = batch['cnn_spec'].unsqueeze(1).to(DEVICE), batch['labels'].to(DEVICE)
            val_preds += cnn_model(x).argmax(-1).cpu().tolist()
            val_labels += labels.cpu().tolist()
    ep_f1 = f1_score(val_labels, val_preds, average='macro')
    print(f'[EffNet] Epoch {epoch:02d} Val F1: {ep_f1:.4f}')
    wandb.log({'effnet_epoch': epoch, 'effnet_val_f1': ep_f1})
    if ep_f1 > effnet_f1:
        effnet_f1 = ep_f1
        torch.save(cnn_model.state_dict(), str(OUT_DIR / 'effnet_best.pt'))
print(f'EfficientNet-B2 Training Complete | Best F1: {effnet_f1:.4f}')
cnn_model.cpu(); torch.cuda.empty_cache()

# Inference (Triple-Ensemble)

In [ ]:
class InferenceDataset(Dataset):
    def __init__(self, paths, offset=0.0, noise_clips=None, snr_db=None):
        self.paths, self.offset, self.snr_db, self.noise_clips = paths, offset, snr_db, noise_clips or []
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        try:
            y, _ = librosa.load(str(self.paths[idx]), sr=SR, mono=True, duration=CLIP_SEC, offset=self.offset)
            if len(y) < CLIP_SAMPLES: y = np.pad(y, (0, CLIP_SAMPLES - len(y)), mode='wrap')
            y = y[:CLIP_SAMPLES]
            if np.max(np.abs(y)) > 0: y /= np.max(np.abs(y))
        except: y = np.zeros(CLIP_SAMPLES, dtype=np.float32)
        if self.snr_db is not None and self.noise_clips:
            y = overlay_noise(y, random.choice(self.noise_clips), self.snr_db)
        
        mert_audio = mert_fe(y.astype(np.float32), sampling_rate=24000, return_tensors='pt')['input_values'].squeeze(0)
        y_16k = resampler_16k(torch.tensor(y.astype(np.float32))).numpy()
        ast_audio = ast_fe(y_16k, sampling_rate=16000, return_tensors='pt')['input_values'].squeeze(0)
        cnn_spec = spectrogram_transform(torch.tensor(y.astype(np.float32))).transpose(0, 1)
        return {'mert_audio': mert_audio, 'ast_audio': ast_audio, 'cnn_spec': cnn_spec}

In [ ]:
model_ast.load_state_dict(torch.load(AST_CHECKPOINT, map_location=DEVICE))
model_mert.load_state_dict(torch.load(MERT_CHECKPOINT, map_location=DEVICE))
cnn_model.load_state_dict(torch.load(str(OUT_DIR / 'effnet_best.pt'), map_location=DEVICE))
model_ast.to(DEVICE).eval(); model_mert.to(DEVICE).eval(); cnn_model.to(DEVICE).eval()
print('Loaded all three best checkpoints to GPU!')

In [ ]:
if ast_f1 > mert_f1:
    w_ast = ast_f1 * 2.0
    w_mert = mert_f1 * 1.0
else:
    w_ast = ast_f1 * 1.0
    w_mert = mert_f1 * 2.0

w_eff = effnet_f1 if effnet_f1 >= 0.70 else 0.0
total_w = w_ast + w_mert + w_eff + 1e-9
w_ast /= total_w; w_mert /= total_w; w_eff /= total_w
print(f'Voting Weights -> AST: {w_ast:.4f} | MERT: {w_mert:.4f} | EffNet: {w_eff:.4f}')

test_df = pd.read_csv(TEST_CSV)
test_paths = [MASHUP_DIR / Path(row['filename']).name for _, row in test_df.iterrows()]
test_ids = test_df['id'].tolist()
prob_accumulator = np.zeros((len(test_ids), 10), dtype=np.float64)
total_passes = 0

for tta_off in TTA_OFFSETS:
    print(f'Clean TTA | offset={tta_off}s')
    dl = DataLoader(InferenceDataset(test_paths, offset=tta_off), batch_size=BATCH_SIZE, num_workers=4, pin_memory=True)
    ptr = 0
    with torch.no_grad():
        for batch in tqdm(dl):
            p_ast = torch.softmax(model_ast(input_values=batch['ast_audio'].to(DEVICE)).logits, -1).cpu().numpy()
            p_mert = torch.softmax(model_mert(batch['mert_audio'].to(DEVICE)), -1).cpu().numpy()
            p_eff = torch.softmax(cnn_model(batch['cnn_spec'].unsqueeze(1).to(DEVICE)), -1).cpu().numpy()
            probs = (w_ast * p_ast) + (w_mert * p_mert) + (w_eff * p_eff)
            prob_accumulator[ptr:ptr+len(probs)] += probs
            ptr += len(probs)
    total_passes += 1

if noise_clips:
    for snr in [8, 12]:
        for tta_off in [0.0, 10.0]:
            print(f'Noise TTA | offset={tta_off}s | SNR={snr}dB')
            dl = DataLoader(InferenceDataset(test_paths, offset=tta_off, noise_clips=noise_clips, snr_db=snr), batch_size=BATCH_SIZE, num_workers=4, pin_memory=True)
            ptr = 0
            with torch.no_grad():
                for batch in tqdm(dl):
                    p_ast = torch.softmax(model_ast(input_values=batch['ast_audio'].to(DEVICE)).logits, -1).cpu().numpy()
                    p_mert = torch.softmax(model_mert(batch['mert_audio'].to(DEVICE)), -1).cpu().numpy()
                    p_eff = torch.softmax(cnn_model(batch['cnn_spec'].unsqueeze(1).to(DEVICE)), -1).cpu().numpy()
                    probs = (w_ast * p_ast) + (w_mert * p_mert) + (w_eff * p_eff)
                    prob_accumulator[ptr:ptr+len(probs)] += probs
                    ptr += len(probs)
            total_passes += 1

predicted_genres = [ID2GENRE[i] for i in (prob_accumulator / total_passes).argmax(axis=1)]
sub = pd.DataFrame({'id': [str(i).zfill(4) for i in test_ids], 'genre': predicted_genres})
sub_path = str(OUT_DIR / 'submission_ast_mert.csv')
sub.to_csv(sub_path, index=False)
print(f'Saved: {sub_path} | Passes: {total_passes}')

In [ ]:
wandb.log({'ast_f1': ast_f1, 'mert_f1': mert_f1, 'effnet_f1': effnet_f1, 'tta_passes': total_passes})
wandb.save(sub_path)
wandb.finish()